# Pilot Data Analysis (2026-02-20)

Analysis of pilot data from the face-flanker memory experiment. 39 subjects: 31 in condition 1 (item recognition), 4 in condition 2 (associative recognition), 4 in condition 3 (valence rating). See `../notes/data-guide.md` for the data dictionary and `../notes/analysis-guide.md` for full analysis rationale and interpretation guidance.

Condition 1 has enough subjects for meaningful group-level summaries and preliminary within-subject statistics. Conditions 2 and 3 remain small (n=4 each) — patterns are descriptive only.

### Design recap

**Study phase (all conditions).** On each trial, a neutral target face appears flanked on both sides by the same emotional face (angry, happy, or neutral). The participant judges the target's gender. The display stays on screen for a fixed 3 seconds regardless of response. The study-phase factorial design crosses target gender (2) × flanker gender (2) × flanker emotion (3) = 12 trial types, each repeated 10 times, for 120 study trials per subject. All faces are from the Chicago Face Database; flankers are restricted to Black and White models to avoid confounding race with emotion.

**Test phase (between-subjects).** Each participant completes one of three test phases:

| Condition | Test task | Display | Trials | Key measure |
|---|---|---|---|---|
| 1 | Item recognition | Single neutral face | 240 (120 old + 120 new) | Old/new judgment |
| 2 | Associative recognition | Three faces (same layout as study) | 120 (60 intact + 60 rearranged) | Same/different judgment |
| 3 | Valence rating | Single neutral face | 120 | 1–9 valence rating |

Condition 2 rearrangement preserves all study-phase factors (target gender, flanker gender, flanker emotion) — only the specific identity pairing changes.

In [1]:
import pandas as pd
from statistics import NormalDist

z = NormalDist().inv_cdf  # equivalent to scipy.stats.norm.ppf

df = pd.read_csv('2026_02_20_pilot_39subj.csv')
df['correct'] = df['correct'].astype('boolean')  # nullable bool (handles NaN from valence trials)

print(f'{len(df)} rows, {df.subject_number.nunique()} subjects')
print(f'Conditions: {df.groupby("condition").subject_number.nunique().to_dict()}')

13080 rows, 39 subjects
Conditions: {1: 31, 2: 4, 3: 4}


## Data Overview

In [2]:
overview = df.groupby(['subject_number', 'condition']).apply(
    lambda g: pd.Series({
        'study_trials': (g.phase == 'study').sum(),
        'test_trials': (g.phase == 'test').sum(),
        'total': len(g),
        'timeouts': g.timed_out.sum()
    })
).reset_index()
print(f'{len(overview)} subjects')
print()
print('Trial counts by condition:')
for cond, g in overview.groupby('condition'):
    n = len(g)
    study_min, study_max = g.study_trials.min(), g.study_trials.max()
    test_min, test_max = g.test_trials.min(), g.test_trials.max()
    timeout_med = g.timeouts.median()
    timeout_max = g.timeouts.max()
    study_str = str(int(study_min)) if study_min == study_max else f'{int(study_min)}-{int(study_max)}'
    test_str = str(int(test_min)) if test_min == test_max else f'{int(test_min)}-{int(test_max)}'
    print(f'  Condition {cond} (n={n}): study={study_str}, test={test_str}, '
          f'timeouts median={timeout_med:.0f} max={timeout_max:.0f}')
print()
print('Subjects with >5 timeouts:')
high_timeout = overview[overview.timeouts > 5]
if len(high_timeout) == 0:
    print('  None')
else:
    print(high_timeout[['subject_number', 'condition', 'timeouts']].to_string(index=False))

39 subjects

Trial counts by condition:
  Condition 1 (n=31): study=120, test=240, timeouts median=1 max=21
  Condition 2 (n=4): study=120, test=120, timeouts median=0 max=3
  Condition 3 (n=4): study=120, test=120, timeouts median=4 max=6

Subjects with >5 timeouts:
 subject_number  condition  timeouts
              6          3         6
              8          1        21
             12          1        11
             14          1        19
             18          1         6
             24          1         6
             26          1        15
             32          1         8


## Study Phase Analysis

The study phase is identical across conditions. Each trial displays a neutral target flanked by two copies of an emotional face; the participant judges the target's gender. The 2 × 2 × 3 factorial (target gender × flanker gender × flanker emotion) produces 12 trial types × 10 reps = 120 trials per subject.

Analyzing study-phase performance checks two things:

1. **Cover task validation** — accuracy should be near ceiling, confirming participants are attending to the faces.
2. **Flanker interference** — RT differences across flanker emotions would indicate the flankers are being processed. Because the display is shown for a fixed 3 seconds regardless of response time, RT differences do not create an encoding-time confound.

In [3]:
study = df[df.phase == 'study'].copy()

# Per-subject accuracy
subj_acc = study.groupby('subject_number').correct.mean()
print(f'Study phase: {len(study)} trials from {study.subject_number.nunique()} subjects')
print(f'Overall accuracy: {study.correct.mean():.1%}')
print(f'Per-subject accuracy: M={subj_acc.mean():.3f}, SD={subj_acc.std():.3f}, '
      f'range=[{subj_acc.min():.3f}, {subj_acc.max():.3f}]')
print()

# Accuracy and RT by flanker emotion
study_by_emotion = study.groupby('flanker_emotion').apply(
    lambda g: pd.Series({
        'N': len(g),
        'accuracy': g.correct.mean(),
        'mean_rt': g.loc[~g.timed_out, 'rt'].mean(),
        'sd_rt': g.loc[~g.timed_out, 'rt'].std(),
        'timeouts': g.timed_out.sum()
    })
).round(3)
print('Accuracy and RT by flanker emotion (pooled across subjects):')
print(study_by_emotion.to_string())

Study phase: 4680 trials from 39 subjects
Overall accuracy: 96.5%
Per-subject accuracy: M=0.965, SD=0.030, range=[0.833, 1.000]

Accuracy and RT by flanker emotion (pooled across subjects):
                      N  accuracy   mean_rt    sd_rt  timeouts
flanker_emotion                                               
angry            1560.0     0.968  1013.971  416.221      17.0
happy            1560.0     0.964  1014.650  437.068      15.0
neutral          1560.0     0.964  1011.310  441.962      14.0


## RQ1: Does Emotional Context Affect Recognition Accuracy? (Condition 1)

At test, condition 1 participants see single neutral faces and judge each as old (studied) or new. The test includes all 120 studied targets plus 120 novel faces. Each studied target was originally flanked by angry, happy, or neutral faces (40 targets per emotion), but the flanker is not shown at test — the question is whether the emotional context at encoding left a trace on memory for the target itself.

**Why signal detection?** A participant who says "old" to every face achieves a perfect hit rate but has zero actual memory — they also say "old" to every new face. Raw hit rates conflate genuine memory with response tendencies. Signal detection theory (SDT) separates the two by computing **d'** (sensitivity — how well the participant can actually distinguish old from new) and **criterion c** (response bias — how willing they are to say "old" when uncertain).

**Reading d':** d' = 0 means chance (guessing). Around 0.5 is weak recognition, 1.0 is moderate, and 2.0 or above is strong. The number represents how well-separated the participant's internal "old" and "new" signals are — higher means old and new faces feel more distinct to them.

**Reading criterion c:** c = 0 is unbiased. Positive c is conservative (when uncertain, tends to say "new" — fewer hits but also fewer false alarms). Negative c is liberal (when uncertain, tends to say "old" — more hits but also more false alarms).

**Why criterion matters here:** A participant might appear to have "better memory" for emotionally-encoded faces simply because they're more willing to call those items "old" (a criterion shift) rather than actually remembering them better (a sensitivity change). Reporting both d' and c lets us distinguish these two explanations.

The false alarm rate is pooled across all new items because new items have no study history and therefore no flanker emotion assignment. This means d' differences across emotions are driven entirely by hit rate differences. With n=31, we compute d' per subject and then summarize across subjects.

In [4]:
def edge_correct(rate, n):
    """Apply Macmillan & Kaplan (1985) edge correction."""
    if rate == 0:
        return 0.5 / n
    if rate == 1:
        return 1 - 0.5 / n
    return rate


c1_test = df[(df.condition == 1) & (df.phase == 'test')].copy()
n_subj_c1 = c1_test.subject_number.nunique()

# Per-subject SDT metrics
rq1_per_subj = []
for subj, sdata in c1_test.groupby('subject_number'):
    new_items = sdata[sdata.stimulus_type == 'new']
    fa_rate_raw = (~new_items.correct).mean()
    fa_n = len(new_items)
    fa_rate = edge_correct(fa_rate_raw, fa_n)

    old_items = sdata[sdata.stimulus_type == 'old']
    for emotion, grp in old_items.groupby('study_flanker_emotion'):
        n = len(grp)
        hit_rate_raw = grp.correct.mean()
        hit_rate = edge_correct(hit_rate_raw, n)
        dprime = z(hit_rate) - z(fa_rate)
        criterion = -0.5 * (z(hit_rate) + z(fa_rate))
        rq1_per_subj.append({
            'subject': subj,
            'emotion': emotion,
            'hit_rate': hit_rate_raw,
            'fa_rate': fa_rate_raw,
            'd_prime': dprime,
            'criterion': criterion
        })

rq1_df = pd.DataFrame(rq1_per_subj)

# Group summary
rq1_summary = rq1_df.groupby('emotion').agg(
    N=('subject', 'count'),
    hit_rate_M=('hit_rate', 'mean'),
    hit_rate_SD=('hit_rate', 'std'),
    fa_rate_M=('fa_rate', 'mean'),
    d_prime_M=('d_prime', 'mean'),
    d_prime_SD=('d_prime', 'std'),
    criterion_M=('criterion', 'mean'),
    criterion_SD=('criterion', 'std'),
).round(3)

print(f'RQ1: Item Recognition (n={n_subj_c1} subjects)')
print(f'Mean FA rate: {rq1_df.fa_rate.mean():.3f} (SD={rq1_df.groupby("subject").fa_rate.first().std():.3f})')
print()
print(rq1_summary.to_string())

RQ1: Item Recognition (n=31 subjects)
Mean FA rate: 0.322 (SD=0.167)

          N  hit_rate_M  hit_rate_SD  fa_rate_M  d_prime_M  d_prime_SD  criterion_M  criterion_SD
emotion                                                                                          
angry    31       0.431        0.174      0.322      0.321       0.349        0.354         0.460
happy    31       0.415        0.178      0.322      0.279       0.382        0.374         0.458
neutral  31       0.427        0.148      0.322      0.318       0.346        0.355         0.425


In [5]:
# Per-subject d' by emotion (for inspecting individual variation)
rq1_pivot = rq1_df.pivot(index='subject', columns='emotion', values='d_prime').round(3)
print("Per-subject d' by flanker emotion:")
print(rq1_pivot.to_string())

Per-subject d' by flanker emotion:
emotion  angry  happy  neutral
subject                       
1        0.778  0.843    0.335
5        0.935  0.681    0.616
8       -0.179  0.338    0.275
9        0.228  0.517   -0.032
10       0.212  0.149    0.212
11       0.796  0.603    0.540
12      -0.023  0.045    0.177
13       0.659  0.827    1.182
14      -0.234 -0.234   -0.108
15       0.344  0.271   -0.064
17      -0.167 -0.167    0.000
18       0.265  0.135    0.068
20       0.532  0.634    0.634
21       0.381  0.448    0.063
22       0.475  0.231    0.398
23       0.105 -0.146    0.441
24       0.384 -0.102    0.636
25       0.030  0.274    0.347
26       0.771  0.704    0.196
27       0.212 -0.337    0.344
28       0.028 -0.053    0.114
29       0.191 -0.271   -0.200
30       0.195  0.195    0.651
31       0.563  0.247    0.047
32       0.063  0.328    0.000
33      -0.049  0.549    0.486
34       0.065  0.193    0.444
35       1.257  1.326    1.190
36       0.302  0.231    0.502
38  

### RQ1 Interpretation

**Overall performance.** Mean d' across all three emotion conditions is around 0.3 — weak recognition, but above chance. Mean FA rate is .32, meaning subjects called roughly a third of new faces "old." These numbers indicate the task is functioning (subjects aren't at floor or ceiling) but memory traces are modest. SDs of d' (~0.35) are about as large as the means, indicating wide individual variation — some subjects show solid recognition (d'>1.0) while others are near or below chance.

**Emotion effect on d'.** The central question: angry d'=0.32, happy d'=0.28, neutral d'=0.32. There is no hint of an emotional enhancement effect. All three conditions are essentially identical, and if anything, happy flankers produced the weakest recognition (0.28) rather than the strongest. This contrasts sharply with the 3-subject pilot, where the single condition 1 subject showed emotional > neutral (d'=0.78/0.84 vs 0.34). That pattern does not replicate across 31 subjects.

**Criterion.** All three conditions show a mildly conservative criterion (~0.35), indicating subjects were somewhat reluctant to say "old" when uncertain. No criterion differences across emotions, consistent with the flat d' pattern.

**Timeouts.** Several subjects (8, 12, 14, 26) had elevated timeout counts (11-21 out of 360 trials). These subjects may have been inattentive or disengaged; their inclusion could be adding noise. Whether to exclude high-timeout subjects is an analysis decision for the full study.

**Bottom line.** With n=31 there is no evidence that emotional flanker context enhances item recognition for the target face. The effect suggested by the 3-subject pilot appears to have been sampling noise. If this pattern holds with the full balanced sample, RQ1 would be a null result.

## RQ2: Are Items Bound More Strongly to Emotional Contexts? (Condition 2)

At test, condition 2 participants see the original three-face display (target + flankers) and judge whether the pairing is intact (same as study) or rearranged. Of the 10 reps per trial type, 5 are tested intact and 5 rearranged, giving 60 intact + 60 rearranged = 120 test trials. Rearranged trials swap flanker identities within the same trial type (preserving target gender, flanker gender, and flanker emotion), so the only cue distinguishing intact from rearranged is memory for the specific identity pairing.

This is a more demanding memory test than item recognition — it requires remembering not just the face, but who it was paired with. d' here measures associative memory strength, not just item familiarity. d' uses the same scale as RQ1 (0 = chance, ~1.0 = moderate, ~2.0 = strong), but values tend to be lower because remembering specific pairings is harder than recognizing individual faces.

Unlike RQ1, false alarm rates are computed per emotion because rearranged trials preserve the flanker emotion. Each emotion condition has its own signal distribution (intact trials) and noise distribution (rearranged trials), allowing d' to adjust for any emotion-specific response biases. There are 20 intact and 20 rearranged trials per emotion per subject.

With only n=4, these results are descriptive — individual variation will dominate.

In [6]:
c2_test = df[(df.condition == 2) & (df.phase == 'test')].copy()
n_subj_c2 = c2_test.subject_number.nunique()

rq2_per_subj = []
for subj, sdata in c2_test.groupby('subject_number'):
    for emotion in sorted(sdata.flanker_emotion.unique()):
        intact = sdata[(sdata.flanker_emotion == emotion) & (sdata.pair_type == 'intact')]
        rearranged = sdata[(sdata.flanker_emotion == emotion) & (sdata.pair_type == 'rearranged')]

        n_intact = len(intact)
        n_rearranged = len(rearranged)
        hit_rate_raw = intact.correct.mean()
        fa_rate_raw = (~rearranged.correct).mean()

        hit_rate = edge_correct(hit_rate_raw, n_intact)
        fa_rate = edge_correct(fa_rate_raw, n_rearranged)

        dprime = z(hit_rate) - z(fa_rate)
        criterion = -0.5 * (z(hit_rate) + z(fa_rate))
        rq2_per_subj.append({
            'subject': subj,
            'emotion': emotion,
            'hit_rate': hit_rate_raw,
            'fa_rate': fa_rate_raw,
            'd_prime': dprime,
            'criterion': criterion
        })

rq2_df = pd.DataFrame(rq2_per_subj)

rq2_summary = rq2_df.groupby('emotion').agg(
    N=('subject', 'count'),
    hit_rate_M=('hit_rate', 'mean'),
    fa_rate_M=('fa_rate', 'mean'),
    d_prime_M=('d_prime', 'mean'),
    d_prime_SD=('d_prime', 'std'),
    criterion_M=('criterion', 'mean'),
    criterion_SD=('criterion', 'std'),
).round(3)

print(f'RQ2: Associative Recognition (n={n_subj_c2} subjects)')
print()
print(rq2_summary.to_string())
print()
print("Per-subject d':")
print(rq2_df.pivot(index='subject', columns='emotion', values='d_prime').round(3).to_string())

RQ2: Associative Recognition (n=4 subjects)

         N  hit_rate_M  fa_rate_M  d_prime_M  d_prime_SD  criterion_M  criterion_SD
emotion                                                                            
angry    4       0.575      0.575     -0.040       0.472       -0.222         0.598
happy    4       0.413      0.562     -0.391       0.287        0.036         0.220
neutral  4       0.500      0.488      0.029       0.199        0.020         0.453

Per-subject d':
emotion  angry  happy  neutral
subject                       
3       -0.167 -0.379    0.253
7       -0.651  0.000    0.139
16       0.271 -0.511   -0.128
37       0.385 -0.674   -0.150


### RQ2 Interpretation

**Performance at floor.** All three emotion conditions produce mean d' near or below zero: angry d'=−0.04, happy d'=−0.39, neutral d'=0.03. Across 4 subjects, no one can reliably distinguish intact from rearranged pairings. This confirms the concern raised in the 3-subject pilot: the associative recognition task may be too difficult.

**Individual variation.** The per-subject table shows wide swings (e.g., subject 37: angry d'=0.39 but happy d'=−0.67). With only 20 intact and 20 rearranged trials per emotion per subject, these values have very wide confidence intervals — a few lucky or unlucky guesses shift d' substantially. No subject shows consistently above-chance performance across all three emotions.

**Response bias.** The angry condition again shows a liberal criterion (c=−0.22), meaning subjects tended to say "same" more often for angry pairings. Happy and neutral are near-unbiased. This echoes the single subject from the 3-subject pilot, but the pattern is weak.

**Design concern.** The within-trial-type rearrangement constraint — which is methodologically correct — removes all non-associative cues (emotion, gender). This makes the task a pure test of identity-level associative memory, but it may be too pure: 4 of 4 subjects perform at chance. If this holds with more subjects, the experiment cannot address RQ2 because there is no signal to modulate. This is worth discussing with the research team — potential mitigations include increasing study time, using fewer study items, or adding a recognition cue.

## RQ3: Do Targets Acquire the Valence of Their Flankers? (Condition 3)

At test, condition 3 participants see each of the 120 studied neutral targets alone (no flanker) and rate its emotional valence on a 1–9 scale (1 = most negative, 5 = neutral, 9 = most positive). This is a direct rating measure — no signal detection is needed and there is no objectively correct answer.

If flanker emotion transfers to the target's mental representation, the predicted ordering of mean ratings is: happy > neutral > angry. Because the faces are objectively neutral, all means are expected to cluster near 5. The question is whether there are reliable between-emotion differences, even if small. Timed-out trials are excluded (no rating was recorded). There are 40 targets per emotion condition per subject.

With only n=4, these results are descriptive.

In [7]:
c3_test = df[(df.condition == 3) & (df.phase == 'test') & ~df.timed_out].copy()
c3_test['rating'] = pd.to_numeric(c3_test['rating'])
n_subj_c3 = c3_test.subject_number.nunique()

# Per-subject mean rating by emotion
rq3_per_subj = c3_test.groupby(['subject_number', 'study_flanker_emotion']).rating.mean().reset_index()

rq3_summary = rq3_per_subj.groupby('study_flanker_emotion').agg(
    N=('subject_number', 'count'),
    mean_rating=('rating', 'mean'),
    sd=('rating', 'std'),
).round(3)

print(f'RQ3: Valence Rating (n={n_subj_c3} subjects)')
timed_out_count = ((df.condition == 3) & (df.phase == 'test') & df.timed_out).sum()
print(f'Timed out trials excluded: {timed_out_count}')
print()
print(rq3_summary.to_string())
print()
print('Per-subject mean rating:')
print(rq3_per_subj.pivot(
    index='subject_number', columns='study_flanker_emotion', values='rating'
).round(3).to_string())

RQ3: Valence Rating (n=4 subjects)
Timed out trials excluded: 6

                       N  mean_rating     sd
study_flanker_emotion                       
angry                  4        5.019  0.198
happy                  4        4.850  0.151
neutral                4        4.902  0.297

Per-subject mean rating:
study_flanker_emotion  angry  happy  neutral
subject_number                              
2                      5.225  4.925    5.289
4                      5.150  4.949    4.900
6                      4.825  4.625    4.568
19                     4.875  4.900    4.850


### RQ3 Interpretation

**Ratings cluster at midpoint.** All three group means fall between 4.85 and 5.02, very close to the scale midpoint (5 = neutral). The predicted ordering (happy > neutral > angry) is not present — angry-context faces were rated slightly most positive (5.02) and happy-context faces least positive (4.85), directionally opposite to the prediction.

**Very low variability.** Between-subject SDs are tiny (0.15–0.30), meaning all 4 subjects gave very similar ratings. Within-subject SDs (not shown but implied by the narrow per-subject means) are also modest. The faces are perceived as neutral and the flanker emotion does not appear to shift that perception.

**Timed out trials.** 6 trials timed out across the 4 subjects. Subject 6 had 6 timeouts — the 3-second window may be slightly tight for the valence rating task, which requires deliberation.

**Bottom line.** With n=4, no evidence of affective transfer from flankers to targets. The effect, if it exists, is smaller than the individual variation in this sample.

## Summary

In [8]:
print('=== Pilot Data Summary (2026-02-20) ===')
print(f'{len(df)} trials, {df.subject_number.nunique()} subjects')
print()
print('Study phase (all subjects):')
print(f'  Overall accuracy: {study.correct.mean():.1%}')
print(f'  Mean RT: {study.loc[~study.timed_out, "rt"].mean():.0f} ms')
print()
print(f'RQ1 - Item Recognition (n={n_subj_c1}):')
for emotion, row in rq1_summary.iterrows():
    print(f"  {emotion:>7}: d'={row['d_prime_M']:.2f} (SD={row['d_prime_SD']:.2f}), "
          f"c={row['criterion_M']:.2f}, hit={row['hit_rate_M']:.2f}")
print()
print(f'RQ2 - Associative Recognition (n={n_subj_c2}):')
for emotion, row in rq2_summary.iterrows():
    print(f"  {emotion:>7}: d'={row['d_prime_M']:.2f} (SD={row['d_prime_SD']:.2f}), "
          f"c={row['criterion_M']:.2f}, hit={row['hit_rate_M']:.2f}, FA={row['fa_rate_M']:.2f}")
print()
print(f'RQ3 - Valence Rating (n={n_subj_c3}):')
for emotion, row in rq3_summary.iterrows():
    print(f'  {emotion:>7}: M={row["mean_rating"]:.2f} (SD={row["sd"]:.2f})')

=== Pilot Data Summary (2026-02-20) ===
13080 trials, 39 subjects

Study phase (all subjects):
  Overall accuracy: 96.5%
  Mean RT: 1013 ms

RQ1 - Item Recognition (n=31):
    angry: d'=0.32 (SD=0.35), c=0.35, hit=0.43
    happy: d'=0.28 (SD=0.38), c=0.37, hit=0.41
  neutral: d'=0.32 (SD=0.35), c=0.35, hit=0.43

RQ2 - Associative Recognition (n=4):
    angry: d'=-0.04 (SD=0.47), c=-0.22, hit=0.57, FA=0.57
    happy: d'=-0.39 (SD=0.29), c=0.04, hit=0.41, FA=0.56
  neutral: d'=0.03 (SD=0.20), c=0.02, hit=0.50, FA=0.49

RQ3 - Valence Rating (n=4):
    angry: M=5.02 (SD=0.20)
    happy: M=4.85 (SD=0.15)
  neutral: M=4.90 (SD=0.30)
